In [6]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
import h5py
import os

In [2]:
# Load labels and pockets
root = '.'
PATH_TO_PROBS = os.path.join(root, "..", "processed", "unidock_docking", "inference_probs")
labels = np.array(pickle.load(open(os.path.join(PATH_TO_PROBS, "success_mols.pkl"), "rb")))
pockets = [i.replace("_bin_01.npz", "") for i in sorted(os.listdir(PATH_TO_PROBS)) if i != "success_mols.pkl"]

print(f"Number of pockets: {len(pockets)}")
print(f"Number of unique molecules: {len(set(labels))}")

Number of pockets: 276
Number of unique molecules: 9556875


In [3]:
# Load probabilities
print("Loading probabilities...")
pocket_to_all_probs = {}
for c, pocket in tqdm(enumerate(pockets)):
    probs = np.load(os.path.join(PATH_TO_PROBS, f"{pocket}_bin_01.npz"))["arr_0"]
    pocket_to_all_probs[pocket] = probs

Loading probabilities...


0it [00:00, ?it/s]

276it [00:54,  5.04it/s]


In [4]:
# Prepare matrix
M = np.column_stack([pocket_to_all_probs[p] for p in pockets])
del pocket_to_all_probs
print(f"Matrix shape: {M.shape}")

print("Normalizing by columns...")

# Calculate means
means_1 = np.mean(M, axis=0, keepdims=True)

# Calculate stds
stds_1 = np.std(M, axis=0, keepdims=True)

# Z-score (1)
M -= means_1
M /= stds_1

print("Normalizing by rows...")

# Calculate means
means_2 = np.mean(M, axis=1, keepdims=True)

# Calculate stds
stds_2 = np.std(M, axis=1, keepdims=True)

# Z-score (2)
M -= means_2
M /= stds_2

Matrix shape: (9556875, 276)
Normalizing by columns...
Normalizing by rows...


In [ ]:
OUT_PATH = os.path.join(root, "..", "processed", "unidock_docking", "inf_probs_norm_matrix.h5")

with h5py.File(OUT_PATH, "w") as f:
    # Main matrix
    f.create_dataset(
        "M",
        data=M,
        compression="gzip",
        compression_opts=9,
    )

    # String dtype for labels & pockets
    str_dt = h5py.string_dtype(encoding="utf-8")

    f.create_dataset(
        "labels",
        data=np.asarray(labels, dtype=str_dt),
        compression="gzip",
        compression_opts=9,
    )

    f.create_dataset(
        "pockets",
        data=np.asarray(pockets, dtype=str_dt),
        compression="gzip",
        compression_opts=9,
    )

print(f"Saved compressed HDF5 to: {OUT_PATH}")